In [7]:
import warnings
warnings.filterwarnings("ignore")

In [8]:
import pandas as pd
from tqdm import tqdm
import json
import numpy as np

Ablation study: All results

In [9]:
models = ["alibaba_cloud/qwen3_235b.tsv", "alibaba_cloud/qwen_qwq.tsv",
          "anthropic/claude_35_haiku.tsv","anthropic/claude_37_sonnet.tsv", 
          "deepseek/deepseek_v3.tsv", "deepseek/deepseek_r1.tsv",
          "openai/gpt35_turbo.tsv", "openai/gpt4o.tsv", "openai/gpt4o_turbo.tsv", "openai/gpt4o_mini.tsv", "openai/o3_mini.tsv", "openai/gpt45_preview.tsv",
          "vertex/gemini_15_pro.tsv", "vertex/gemini_25_flash.tsv",
          "xai/grok2.tsv", 
          "meta/llama31_8b.tsv", "meta/llama31_405b.tsv", "meta/llama32_3b.tsv", "meta/llama33_70b.tsv", "meta/llama4_maverick.tsv",
          "mistral/mistral_7b.tsv",
          "microsoft/phi4.tsv"]
model_names = ["Qwen 3 235b", "Qwen QwQ 32b",
               "Claude 3.5 Haiku", "Claude 3.7 Sonnet", 
               "DeepSeek-V3", "DeepSeek-R1",
               "GPT 3.5 Turbo", "GPT 4o", "GPT 4o Turbo", "GPT 4o Mini", "o3 Mini", "GPT 4.5 Preview",
               "Gemini 1.5 Pro", "Gemini 2.5 Flash",
               "Grok 2", 
               "Llama 3.1:8b", "Llama 3.1:405b", "Llama 3.2:3b", "Llama 3.3:70b", "Llama 4 Maverick",
               "Mistral:7b",
               "Phi 4"]

In [10]:
all_results_ensemble = []
length_of_list_of_predictions_for_each_model = []

for model in models:
    count = 0

    df = pd.read_csv("../../data/processed/models/" + model, delimiter="\t")
    #print(df[" array_of_pol_pos "])
    for index, row in df.iterrows():
        predictions = eval(row[" array_of_pol_pos "])
        all_results_ensemble.extend(predictions)
        count += len(predictions)

    length_of_list_of_predictions_for_each_model.append(count)

In [11]:
np.mean(all_results_ensemble)

np.float64(42.961739054213965)

In [12]:
np.std(all_results_ensemble)

np.float64(21.509693771037284)

In [13]:
len(length_of_list_of_predictions_for_each_model)

22

In [14]:
print(f"Ensemble mean: {np.mean(all_results_ensemble):.2f}")
print(f"Ensemble std: {np.std(all_results_ensemble):.2f}")

count = 0 

for index, (len_of_predictions, model_name) in enumerate(zip(length_of_list_of_predictions_for_each_model, model_names)):    

    temp_results = all_results_ensemble[:count] + all_results_ensemble[count+len_of_predictions:]
    print(f"{model_name}: ", end=" ")
    print(f"Mean= {np.mean(temp_results):.2f}, ", end=" ")
    print(f"Std= {np.std(temp_results):.2f}")
    #print(f"Correct len() of reduced ensemble: {(len(all_results_ensemble) - len_of_predictions)}")
    #print(f"len() of tmp_results: {len(temp_results)}")
    print("---------------")
    count += len_of_predictions



if count == len(all_results_ensemble):
    print("We've correctly counted the different number of preds!!!")

Ensemble mean: 42.96
Ensemble std: 21.51
Qwen 3 235b:  Mean= 42.93,  Std= 21.59
---------------
Qwen QwQ 32b:  Mean= 42.88,  Std= 21.52
---------------
Claude 3.5 Haiku:  Mean= 42.42,  Std= 22.05
---------------
Claude 3.7 Sonnet:  Mean= 42.90,  Std= 21.59
---------------
DeepSeek-V3:  Mean= 42.72,  Std= 21.53
---------------
DeepSeek-R1:  Mean= 42.81,  Std= 21.40
---------------
GPT 3.5 Turbo:  Mean= 42.90,  Std= 21.62
---------------
GPT 4o:  Mean= 42.99,  Std= 21.56
---------------
GPT 4o Turbo:  Mean= 43.01,  Std= 21.48
---------------
GPT 4o Mini:  Mean= 42.80,  Std= 22.01
---------------
o3 Mini:  Mean= 42.90,  Std= 21.68
---------------
GPT 4.5 Preview:  Mean= 42.96,  Std= 21.56
---------------
Gemini 1.5 Pro:  Mean= 43.00,  Std= 21.45
---------------
Gemini 2.5 Flash:  Mean= 42.94,  Std= 21.60
---------------
Grok 2:  Mean= 43.00,  Std= 21.74
---------------
Llama 3.1:8b:  Mean= 42.04,  Std= 21.83
---------------
Llama 3.1:405b:  Mean= 42.82,  Std= 21.62
---------------
Llama 3

# Creating a reduced ensemble (when NA count > len(predicitons))

When looking at the data, we noticed that less capable models (with fewer parameters) struggled to differentiate between when a proposition and locution were (a)political. Given that propositions and locutions are essentially clauses, we make the assumption that there will be more datapoints that are *not applicable* than *political*. To identify models which were not able to differentiate between political / apolitical propositions and locutions, we counted the number of times that a model scored a node as either political or NA. Then, if the NA count was less than the number of political scores, then it was removed from the second ensemble.

In [15]:
second_ensemble = []
second_ensemble_models = []
second_ensemble_length_of_list_of_predictions_for_each_model = []

for i, (model, model_name) in enumerate(zip(models, model_names)):
    df = pd.read_csv("../../data/processed/models/" + model, delimiter="\t")
    na_count = 0
    results = []
    for index, row in df.iterrows():
        predictions = eval(row[" array_of_pol_pos "])
        results.extend(predictions)

        na_count += row[" NA_count "]

    if na_count > len(results):
        second_ensemble.extend(results)
        count = len(results)
        second_ensemble_models.append(model_name)
        print(f"{model_name} included.")
        second_ensemble_length_of_list_of_predictions_for_each_model.append(count)

len(second_ensemble)

Qwen 3 235b included.
Claude 3.7 Sonnet included.
DeepSeek-V3 included.
DeepSeek-R1 included.
GPT 4o included.
GPT 4o Turbo included.
GPT 4.5 Preview included.
Gemini 1.5 Pro included.
Gemini 2.5 Flash included.
Grok 2 included.
Llama 4 Maverick included.
Phi 4 included.


439060

In [16]:
print(f"Ensemble mean: {np.mean(second_ensemble):.2f}")
print(f"Ensemble std: {np.std(second_ensemble):.2f}")

count = 0 

for index, (len_of_predictions, model_name) in enumerate(zip(second_ensemble_length_of_list_of_predictions_for_each_model, second_ensemble_models)):    

    temp_results = second_ensemble[:count] + second_ensemble[count+len_of_predictions:]
    print(f"{model_name}: ", end=" ")
    print(f"Mean= {np.mean(temp_results):.2f}, ", end=" ")
    print(f"Std= {np.std(temp_results):.2f}")
    print("---------------")
    print(len_of_predictions)
    count += len_of_predictions



if count == len(second_ensemble):
    print("We've correctly counted the different number of preds!!!")

print(count)

Ensemble mean: 43.24
Ensemble std: 20.70
Qwen 3 235b:  Mean= 43.17,  Std= 20.88
---------------
50137
Claude 3.7 Sonnet:  Mean= 43.07,  Std= 20.91
---------------
20168
DeepSeek-V3:  Mean= 42.50,  Std= 20.70
---------------
37531
DeepSeek-R1:  Mean= 42.78,  Std= 20.23
---------------
42249
GPT 4o:  Mean= 43.35,  Std= 20.79
---------------
37111
GPT 4o Turbo:  Mean= 43.40,  Std= 20.56
---------------
13117
GPT 4.5 Preview:  Mean= 43.26,  Std= 20.80
---------------
26049
Gemini 1.5 Pro:  Mean= 43.38,  Std= 20.44
---------------
31891
Gemini 2.5 Flash:  Mean= 43.19,  Std= 20.91
---------------
39799
Grok 2:  Mean= 43.40,  Std= 21.39
---------------
41112
Llama 4 Maverick:  Mean= 43.98,  Std= 20.54
---------------
47051
Phi 4:  Mean= 43.43,  Std= 20.15
---------------
52845
We've correctly counted the different number of preds!!!
439060


# Reasoning models

In [18]:
reasoning_models = ["alibaba_cloud/qwen3_235b.tsv", "alibaba_cloud/qwen_qwq.tsv",
          "anthropic/claude_37_sonnet.tsv", 
           "deepseek/deepseek_r1.tsv",
          "openai/o3_mini.tsv"]
reasoning_model_names = ["Qwen 3 235b", "Qwen QwQ 32b",
                "Claude 3.7 Sonnet", 
               "DeepSeek-R1",
               "o3 Mini"]

In [25]:
reasoning_ensemble = []
reasoning_ensemble_length_of_list_of_predictions_for_each_model = []

for model in reasoning_models:
    count = 0
    df = pd.read_csv("../../data/processed/models/" + model, delimiter="\t")
    for index, row in df.iterrows():
        predictions = eval(row[" array_of_pol_pos "])
        reasoning_ensemble.extend(predictions)
        count += len(predictions)

    reasoning_ensemble_length_of_list_of_predictions_for_each_model.append(count)

len(reasoning_ensemble)

247908

In [26]:
print(f"Ensemble mean: {np.mean(reasoning_ensemble):.2f}")
print(f"Ensemble std: {np.std(reasoning_ensemble):.2f}")

count = 0 

for index, (len_of_predictions, model_name) in enumerate(zip(reasoning_ensemble_length_of_list_of_predictions_for_each_model, reasoning_model_names)):    

    temp_results = reasoning_ensemble[:count] + reasoning_ensemble[count+len_of_predictions:]
    print(f"{model_name}: ", end=" ")
    print(f"Mean= {np.mean(temp_results):.2f}, ", end=" ")
    print(f"Std= {np.std(temp_results):.2f}")
    print("---------------")
    print(len_of_predictions)
    count += len_of_predictions



if count == len(reasoning_ensemble):
    print("We've correctly counted the different number of preds!!!")

print(count)

Ensemble mean: 44.93
Ensemble std: 20.20
Qwen 3 235b:  Mean= 45.21,  Std= 20.43
---------------
50137
Qwen QwQ 32b:  Mean= 45.03,  Std= 19.88
---------------
61390
Claude 3.7 Sonnet:  Mean= 44.77,  Std= 20.56
---------------
20168
DeepSeek-R1:  Mean= 44.39,  Std= 19.22
---------------
42249
o3 Mini:  Mean= 45.36,  Std= 20.89
---------------
73964
We've correctly counted the different number of preds!!!
247908


We can see the following from the above:

- Removing both qwen3 and qwen qwq makes the ensemble average closer to 50; thus, both of the alibaba_cloud models contribute more left-leaning predictions to the reasoning ensemble mean. 

- Removing Claude 3.7 Sonnet changes the reasoning ensemble mean to a more left-wing average, which means that Claude 3.7 Sonnet's predictions were, on average, to the right of the ensemble mean.

- DeepSeek-R1's average contribution is similar as Claude's. 

- o3 mini produced, on average, more left wing predictions such that it's removal from the reasoning ensemble made the mean closer to the centre. 

All reasoning models predicted that, on average, all propositions and locutions were left of centre.

In [27]:
list_of_str = ["claude_35_haiku_pol_pos_array", "claude_35_haiku_pol_pos_mean", "claude_35_haiku_pol_pos_variance", "claude_35_haiku_pol_pos_standard_deviation", "claude_35_haiku_pol_pos_NA_count", "claude_35_haiku_pol_pos_probability_of_NA", "claude_37_sonnet_pol_pos_array", "claude_37_sonnet_pol_pos_mean", "claude_37_sonnet_pol_pos_variance", "claude_37_sonnet_pol_pos_standard_deviation", "claude_37_sonnet_pol_pos_NA_count", "claude_37_sonnet_pol_pos_probability_of_NA", "deepseek_v3_pol_pos_array", "deepseek_v3_pol_pos_mean", "deepseek_v3_pol_pos_variance", "deepseek_v3_pol_pos_standard_deviation", "deepseek_v3_pol_pos_NA_count", "deepseek_v3_pol_pos_probability_of_NA", "gpt35_turbo_pol_pos_array", "gpt35_turbo_pol_pos_mean", "gpt35_turbo_pol_pos_variance", "gpt35_turbo_pol_pos_standard_deviation", "gpt35_turbo_pol_pos_NA_count", "gpt35_turbo_pol_pos_probability_of_NA", "gpt4o_pol_pos_array", "gpt4o_pol_pos_mean", "gpt4o_pol_pos_variance", "gpt4o_pol_pos_standard_deviation", "gpt4o_pol_pos_NA_count", "gpt4o_pol_pos_probability_of_NA", "gpt4o_turbo_pol_pos_array", "gpt4o_turbo_pol_pos_mean", "gpt4o_turbo_pol_pos_variance", "gpt4o_turbo_pol_pos_standard_deviation", "gpt4o_turbo_pol_pos_NA_count", "gpt4o_turbo_pol_pos_probability_of_NA", "gpt4o_mini_pol_pos_array", "gpt4o_mini_pol_pos_mean", "gpt4o_mini_pol_pos_variance", "gpt4o_mini_pol_pos_standard_deviation", "gpt4o_mini_pol_pos_NA_count", "gpt4o_mini_pol_pos_probability_of_NA", "o3_mini_pol_pos_array", "o3_mini_pol_pos_mean", "o3_mini_pol_pos_variance", "o3_mini_pol_pos_standard_deviation", "o3_mini_pol_pos_NA_count", "o3_mini_pol_pos_probability_of_NA", "gemini_15_pro_pol_pos_array", "gemini_15_pro_pol_pos_mean", "gemini_15_pro_pol_pos_variance", "gemini_15_pro_pol_pos_standard_deviation", "gemini_15_pro_pol_pos_NA_count", "gemini_15_pro_pol_pos_probability_of_NA", "grok2_pol_pos_array", "grok2_pol_pos_mean", "grok2_pol_pos_variance", "grok2_pol_pos_standard_deviation", "grok2_pol_pos_NA_count", "grok2_pol_pos_probability_of_NA", "llama31_8b_pol_pos_array", "llama31_8b_pol_pos_mean", "llama31_8b_pol_pos_variance", "llama31_8b_pol_pos_standard_deviation", "llama31_8b_pol_pos_NA_count", "llama31_8b_pol_pos_probability_of_NA", "llama32_3b_pol_pos_array", "llama32_3b_pol_pos_mean", "llama32_3b_pol_pos_variance", "llama32_3b_pol_pos_standard_deviation", "llama32_3b_pol_pos_NA_count", "llama32_3b_pol_pos_probability_of_NA", "mistral_7b_pol_pos_array", "mistral_7b_pol_pos_mean", "mistral_7b_pol_pos_variance", "mistral_7b_pol_pos_standard_deviation", "mistral_7b_pol_pos_NA_count", "mistral_7b_pol_pos_probability_of_NA", "ensemble_all_results_pol_pos_array", "ensemble_all_results_pol_pos_mean", "ensemble_all_results_pol_pos_variance", "ensemble_all_results_pol_pos_standard_deviation", "ensemble_all_results_pol_pos_NA_count", "ensemble_reduced_results_pol_pos_array", "ensemble_reduced_results_pol_pos_mean", "ensemble_reduced_results_pol_pos_variance", "ensemble_reduced_results_pol_pos_standard_deviation", "ensemble_reduced_results_pol_pos_NA_count", "ensemble_reduced_results_pol_pos_probability_of_NA"]

In [29]:
new_list = []
for string in list_of_str:
    new_list.append("n."+string)
new_list

['n.claude_35_haiku_pol_pos_array',
 'n.claude_35_haiku_pol_pos_mean',
 'n.claude_35_haiku_pol_pos_variance',
 'n.claude_35_haiku_pol_pos_standard_deviation',
 'n.claude_35_haiku_pol_pos_NA_count',
 'n.claude_35_haiku_pol_pos_probability_of_NA',
 'n.claude_37_sonnet_pol_pos_array',
 'n.claude_37_sonnet_pol_pos_mean',
 'n.claude_37_sonnet_pol_pos_variance',
 'n.claude_37_sonnet_pol_pos_standard_deviation',
 'n.claude_37_sonnet_pol_pos_NA_count',
 'n.claude_37_sonnet_pol_pos_probability_of_NA',
 'n.deepseek_v3_pol_pos_array',
 'n.deepseek_v3_pol_pos_mean',
 'n.deepseek_v3_pol_pos_variance',
 'n.deepseek_v3_pol_pos_standard_deviation',
 'n.deepseek_v3_pol_pos_NA_count',
 'n.deepseek_v3_pol_pos_probability_of_NA',
 'n.gpt35_turbo_pol_pos_array',
 'n.gpt35_turbo_pol_pos_mean',
 'n.gpt35_turbo_pol_pos_variance',
 'n.gpt35_turbo_pol_pos_standard_deviation',
 'n.gpt35_turbo_pol_pos_NA_count',
 'n.gpt35_turbo_pol_pos_probability_of_NA',
 'n.gpt4o_pol_pos_array',
 'n.gpt4o_pol_pos_mean',
 'n.gpt

In [30]:
for string in new_list:
    print(string+", ", end=" ")

n.claude_35_haiku_pol_pos_array,  n.claude_35_haiku_pol_pos_mean,  n.claude_35_haiku_pol_pos_variance,  n.claude_35_haiku_pol_pos_standard_deviation,  n.claude_35_haiku_pol_pos_NA_count,  n.claude_35_haiku_pol_pos_probability_of_NA,  n.claude_37_sonnet_pol_pos_array,  n.claude_37_sonnet_pol_pos_mean,  n.claude_37_sonnet_pol_pos_variance,  n.claude_37_sonnet_pol_pos_standard_deviation,  n.claude_37_sonnet_pol_pos_NA_count,  n.claude_37_sonnet_pol_pos_probability_of_NA,  n.deepseek_v3_pol_pos_array,  n.deepseek_v3_pol_pos_mean,  n.deepseek_v3_pol_pos_variance,  n.deepseek_v3_pol_pos_standard_deviation,  n.deepseek_v3_pol_pos_NA_count,  n.deepseek_v3_pol_pos_probability_of_NA,  n.gpt35_turbo_pol_pos_array,  n.gpt35_turbo_pol_pos_mean,  n.gpt35_turbo_pol_pos_variance,  n.gpt35_turbo_pol_pos_standard_deviation,  n.gpt35_turbo_pol_pos_NA_count,  n.gpt35_turbo_pol_pos_probability_of_NA,  n.gpt4o_pol_pos_array,  n.gpt4o_pol_pos_mean,  n.gpt4o_pol_pos_variance,  n.gpt4o_pol_pos_standard_deviati